# 🎬 AIC 2026 Multimedia Retrieval System - Kaggle GPU Executor
Notebook tự động chạy thử nghiệm quy trình **Offline Indexing (Phase 1)** trên Kaggle GPU (T4 / P100).
- Tự động kích hoạt GPU CUDA & FP16 Half-Precision.
- Tự động nạp Gemini API Key bảo mật qua Kaggle Secrets hoặc Environment Variable.

In [ ]:
# CELL 1: Clone Repository mã nguồn từ GitHub của bạn
!git clone https://github.com/DntdToM/aic2026_retrieval_project.git
%cd aic2026_retrieval_project

In [ ]:
# CELL 2: Cài đặt các thư viện hệ thống cần thiết (FFmpeg, OpenCV dependencies)
!apt-get update && apt-get install -y ffmpeg libsm6 libxext6

In [ ]:
# CELL 3: Cài đặt các thư viện Python từ requirements.txt
!pip install -r requirements.txt

In [ ]:
# CELL 4: Cấu hình Gemini API Key, Tải Video Dataset từ Google Drive & Tải Weights Mô hình
import os
import zipfile
import gdown

# 1. Thiết lập Gemini API Key (Ưu tiên Kaggle Secrets bảo mật, fallback điền trực tiếp)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
    print("🔑 Đã nạp Gemini API Key từ Kaggle Secrets thành công!")
except Exception:
    # Điền trực tiếp API Key của bạn bên dưới nếu không dùng Kaggle Secrets:
    os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"
    print("🔑 Đã nạp Gemini API Key từ Environment Variable!")

# 2. Tải Dataset Video ZIP từ Google Drive (Nếu có File ID)
GDRIVE_FILE_ID = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE"  # Thay ID file zip từ Google Drive của bạn
TARGET_DIR = "data/official_videos"
os.makedirs(TARGET_DIR, exist_ok=True)

if GDRIVE_FILE_ID != "YOUR_GOOGLE_DRIVE_FILE_ID_HERE":
    print(f"📥 Đang tải Dataset video từ Google Drive (ID: {GDRIVE_FILE_ID})...")
    zip_path = "videos_dataset.zip"
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", zip_path, quiet=False)
    print(f"📦 Đang giải nén {zip_path} vào {TARGET_DIR}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(TARGET_DIR)
    print("✅ Giải nén video thành công!")

# 3. Tự động kiểm tra và tải Weights Mô hình nếu chưa có sẵn trong models/
if not os.path.exists("models/siglip-base-patch16-224") or not os.path.exists("models/bge-m3"):
    print("📦 Đang tự động tải Weights các mô hình về thư mục local models/...")
    !python scripts/download_models.py
else:
    print("✅ Các mô hình đã sẵn sàng trong thư mục models/!")

In [ ]:
# CELL 5: Khởi chạy toàn bộ Offline Indexing Pipeline
!python run_pipeline.py

In [ ]:
# CELL 6: Zip toàn bộ thư mục kết quả processed_data/ thành output_data.zip để tải về
import os
import shutil

OUTPUT_ZIP_NAME = "output_data"
SOURCE_DIR = "processed_data"

if os.path.exists(SOURCE_DIR):
    print(f"Đang nén toàn bộ thư mục '{SOURCE_DIR}' thành '{OUTPUT_ZIP_NAME}.zip'...")
    shutil.make_archive(OUTPUT_ZIP_NAME, 'zip', SOURCE_DIR)
    zip_file_path = f"{OUTPUT_ZIP_NAME}.zip"
    size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)
    print(f"✅ Đã nén thành công file: {zip_file_path} ({size_mb:.2f} MB)")
    print("📥 Bạn có thể tải file 'output_data.zip' trực tiếp ở mục Output bên phải màn hình Kaggle!")
else:
    print(f"❌ Không tìm thấy thư mục {SOURCE_DIR}.")